In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
train.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [4]:
test.describe()

,id
count,500.000000
mean,250.500000
std,144.481833
min,1.000000
25%,125.750000
50%,250.500000
75%,375.250000
max,500.000000


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      500 non-null    int64 
 1   prompt  500 non-null    object
 2   A       500 non-null    object
 3   B       500 non-null    object
 4   C       500 non-null    object
 5   D       500 non-null    object
 6   E       500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


In [7]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [9]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
OPTIONS = ['A', 'B', 'C', 'D', 'E']

In [10]:
print("Columns:", df.columns.tolist())
print("Shape  :", df.shape)
print(df.head(2))

Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Shape  : (2000, 8)
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   C  \
0  Martin Heidegger does not believe in the exist...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   D  \
0  Martin Heidegger believes that the relationshi...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   E answer  
0

In [11]:
# Q1 — Frequency distribution of correct answers
freq = df['answer'].value_counts()
most_freq_option  = freq.idxmax()
least_freq_option = freq.idxmin()


In [12]:
print("=== Q1: Answer Frequency Distribution ===")
print(freq.to_string())
print(f"\nMost frequent : {most_freq_option} → {freq[most_freq_option]}")
print(f"Least frequent: {least_freq_option} → {freq[least_freq_option]}")
print(f"Sum (most + least): {freq[most_freq_option] + freq[least_freq_option]}")

=== Q1: Answer Frequency Distribution ===
answer
B    490
C    459
A    369
D    358
E    324

Most frequent : B → 490
Least frequent: E → 324
Sum (most + least): 814


In [13]:
# Q2 — Vocabulary size after lowercasing + removing punctuation
import string
def clean_text(text: str) -> str:
    """Lowercase and strip all string.punctuation characters."""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

cleaned_prompts = df['prompt'].apply(clean_text)

vocab = set()
for text in cleaned_prompts:
    vocab.update(text.split())

print(f"\n=== Q2: Vocabulary Size (cleaned prompts) ===")
print(f"Unique words: {len(vocab)}")


=== Q2: Vocabulary Size (cleaned prompts) ===
Unique words: 859


In [14]:
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text
 
df["clean_prompt"] = df["prompt"].apply(clean_text)
 
all_words = []
for text in df["clean_prompt"]:
    all_words.extend(text.split())
 
vocab = set(all_words)
print("\n── Q2 ──")
print(f"Total unique words (vocab size): {len(vocab)}")


── Q2 ──
Total unique words (vocab size): 859


In [15]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [16]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
row1 = df[df["id"] == 1].iloc[0]           
words_row1 = row1["clean_prompt"].split()
filtered_row1 = [w for w in words_row1 if w not in ENGLISH_STOP_WORDS]
 
print("\n── Q3 ──")
print(f"Words in Row 1 prompt (cleaned)       : {len(words_row1)}")
print(f"Words after stop-word removal (Q3 ans): {len(filtered_row1)}")


── Q3 ──
Words in Row 1 prompt (cleaned)       : 22
Words after stop-word removal (Q3 ans): 13


In [17]:
# Q4 — TF-IDF vocabulary size (fitted on prompt + all 5 options per row)

from sklearn.feature_extraction.text import TfidfVectorizer
combined_texts = []
for _, row in df.iterrows():
    combined = ' '.join([str(row['prompt'])] + [str(row[opt]) for opt in OPTIONS])
    combined_texts.append(combined)

tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(combined_texts)

print(f"\n=== Q4: TF-IDF Feature Columns (vocabulary size) ===")
print(f"Features: {len(tfidf.vocabulary_)}")


=== Q4: TF-IDF Feature Columns (vocabulary size) ===
Features: 2762


In [18]:
# Q5 — Cosine similarity: Row ID 1, Prompt vs Option A

from sklearn.metrics.pairwise import cosine_similarity
row1     = df[df['id'] == 1].iloc[0]
p_vec    = tfidf.transform([str(row1['prompt'])])
a_vec    = tfidf.transform([str(row1['A'])])
sim_q5   = cosine_similarity(p_vec, a_vec)[0][0]

print(f"\n=== Q5: Cosine Similarity (Row ID 1, Prompt vs A) ===")
print(f"Similarity: {round(sim_q5, 4)}")



=== Q5: Cosine Similarity (Row ID 1, Prompt vs A) ===
Similarity: 0.272


In [19]:
# Q6 — % of rows where highest cosine-sim option = correct answer

def get_ranked_options(row, vectorizer) -> list:
    """Return options sorted by cosine similarity to the prompt (desc)."""
    p = vectorizer.transform([str(row['prompt'])])
    sims = {opt: cosine_similarity(p, vectorizer.transform([str(row[opt])]))[0][0]
            for opt in OPTIONS}
    return sorted(sims, key=sims.get, reverse=True)

correct_count = sum(
    1 for _, row in df.iterrows()
    if get_ranked_options(row, tfidf)[0] == row['answer']
)
pct = correct_count / len(df) * 100

print(f"\n=== Q6: TF-IDF Top-1 Match Rate ===")
print(f"{correct_count}/{len(df)} = {round(pct, 2)}%")


=== Q6: TF-IDF Top-1 Match Rate ===
271/2000 = 13.55%


In [20]:
# MAP@3 helper

def map_at_3(truth: str, preds: list) -> float:
    """Mean Average Precision at 3 for a single question."""
    for i, p in enumerate(preds[:3]):
        if p == truth:
            return 1.0 / (i + 1)
    return 0.0


In [21]:
# Q7 — MAP@3: truth=C, preds=[C, A, B]

score_q7 = map_at_3('C', ['C', 'A', 'B'])
print(f"\n=== Q7: MAP@3 (truth=C, preds=[C,A,B]) ===")
print(f"Score: {score_q7}")



=== Q7: MAP@3 (truth=C, preds=[C,A,B]) ===
Score: 1.0


In [22]:
# Q8 — MAP@3: truth=B, preds=[D, B, E]

score_q8 = map_at_3('B', ['D', 'B', 'E'])
print(f"\n=== Q8: MAP@3 (truth=B, preds=[D,B,E]) ===")
print(f"Score: {score_q8}")


=== Q8: MAP@3 (truth=B, preds=[D,B,E]) ===
Score: 0.5


In [23]:
# Q9 — Majority Class Baseline MAP@3

top3_majority = freq.index[:3].tolist()   # ['B', 'C', 'A']
majority_scores = [map_at_3(row['answer'], top3_majority) for _, row in df.iterrows()]

print(f"\n=== Q9: Majority Class Baseline ===")
print(f"Static prediction: {top3_majority}")
print(f"Overall MAP@3: {round(np.mean(majority_scores), 4)}")


=== Q9: Majority Class Baseline ===
Static prediction: ['B', 'C', 'A']
Overall MAP@3: 0.4212


In [24]:
# Q10 — TF-IDF Pipeline MAP@3

tfidf_scores = []
for _, row in df.iterrows():
    ranked = get_ranked_options(row, tfidf)
    tfidf_scores.append(map_at_3(row['answer'], ranked[:3]))

print(f"\n=== Q10: TF-IDF Pipeline MAP@3 ===")
print(f"Average MAP@3: {round(np.mean(tfidf_scores), 4)}")


=== Q10: TF-IDF Pipeline MAP@3 ===
Average MAP@3: 0.2962
